# Playoff odds: calibration on the 2025 season

Runs the playoff odds model (`docs/METRICS_SPEC.md` section 8, `config.yaml` `metrics.playoff_odds`) as of each week of this league's
2025 regular season, using only the weeks played by then, and compares the predicted odds with who actually made the playoffs.

- **Brier score**: mean of (predicted probability − outcome)², per week and overall; lower is better. Baselines: every team at 6/12 = 50%,
  and "in the top 6 right now" (100% or 0%).
- **Reliability table**: predictions grouped by predicted probability, with the share that actually made it.
- **Caveats.** One season gives 12 teams × 12 weeks = 144 predictions but only 12 outcomes, and each team's predictions across weeks are
  strongly correlated, so the intervals below resample teams, not predictions. `shrink_weeks` (6) was itself calibrated on 2025's scores,
  so the sensitivity table at the end is in-sample.
- **Data.** The 2025 league (`1243747994637963265`) is downloaded once into `data/raw/2025/` (gitignored) by the project's own extract step,
  about 45 API calls; later runs reuse it. Teams are shown by `roster_id` only. Commit this notebook without outputs.

In [ ]:
import numpy as np
import pandas as pd

from sleeper_dash import extract, transform, validate
from sleeper_dash.config import Config, load_config
from sleeper_dash.metrics import playoff_odds as po

pd.set_option("display.width", 160)
LEAGUE_2025, SEASON = "1243747994637963265", 2025
PARAMS = load_config().metrics["playoff_odds"]
PARAMS

## 1. The 2025 season

In [ ]:
raw = extract.RAW_DIR / str(SEASON)
if not (raw / "winners_bracket.json").exists():  # download once; a full refresh of the finished season
    extract.extract(Config(league_id=LEAGUE_2025, season=SEASON, season_start_dates={}, metrics={}))
league, rosters = transform.read_raw(SEASON, "league.json"), transform.read_raw(SEASON, "rosters.json")
matchups = transform.read_matchups(SEASON)
team_weeks = transform.build_team_weeks(league, matchups)
schedule = transform.build_schedule(league, matchups, {})
bracket = transform.build_winners_bracket(league, transform.read_raw(SEASON, "winners_bracket.json"))
settings = league["settings"]
print({k: settings[k] for k in ["num_teams", "playoff_week_start", "playoff_teams", "playoff_round_type", "playoff_seed_type", "league_average_match"]})
print("weeks:", sorted(team_weeks["week"].unique()))

In [ ]:
# The tables reconcile with Sleeper, and the final standings give Sleeper's real 2025 bracket.
for check in [validate.check_records(team_weeks, rosters, league),
              validate.check_points_for_against(team_weeks, rosters),
              validate.check_seeding_matches_sleeper(bracket, team_weeks, league)]:
    print("PASS" if check.passed else "FAIL", check.name, "-", check.detail)

REGULAR = settings["playoff_week_start"] - 1
final = po.standings_through(team_weeks, REGULAR)
seeds = po.seeding(final)
final["seed"] = [seeds.index(r) + 1 for r in final.index]
final["made_playoffs"] = final["seed"] <= 6
final["got_bye"] = final["seed"] <= 2
final.sort_values("seed")[["wins", "losses", "points_for", "seed", "made_playoffs", "got_bye"]]

## 2. The model as of each week (weeks 3–14)

In [ ]:
odds = po.build_playoff_odds(team_weeks, schedule, league, PARAMS)
odds = odds.join(final[["seed", "made_playoffs", "got_bye"]], on="roster_id")
trajectory = odds.pivot(index="roster_id", columns="week", values="p_playoffs").mul(100).round(0).astype(int)
trajectory.insert(0, "final seed", final["seed"])
trajectory.sort_values("final seed")  # playoff odds (%) after each week, by final seed

## 3. Brier scores

In [ ]:
def brier(p, outcome):
    return float(np.mean((np.asarray(p, float) - np.asarray(outcome, float)) ** 2))

def current_top6(week):
    table = po.standings_through(team_weeks, week)
    order = po.seeding(table)
    return pd.Series({r: float(order.index(r) < 6) for r in table.index})

odds["now_top6"] = [current_top6(w)[r] for w, r in zip(odds["week"], odds["roster_id"])]
odds["now_top2"] = [float(po.seeding(po.standings_through(team_weeks, w)).index(r) < 2) for w, r in zip(odds["week"], odds["roster_id"])]

by_week = odds.groupby("week").apply(lambda g: pd.Series({
    "model": brier(g["p_playoffs"], g["made_playoffs"]),
    "everyone 50%": brier(np.full(len(g), 0.5), g["made_playoffs"]),
    "top 6 now": brier(g["now_top6"], g["made_playoffs"]),
    "bye: model": brier(g["p_bye"], g["got_bye"]),
    "bye: everyone 2/12": brier(np.full(len(g), 2 / 12), g["got_bye"]),
    "bye: top 2 now": brier(g["now_top2"], g["got_bye"]),
}), include_groups=False)
overall = pd.DataFrame({
    "model": [brier(odds["p_playoffs"], odds["made_playoffs"])],
    "everyone 50%": [0.25],
    "top 6 now": [brier(odds["now_top6"], odds["made_playoffs"])],
    "bye: model": [brier(odds["p_bye"], odds["got_bye"])],
    "bye: everyone 2/12": [brier(np.full(len(odds), 2 / 12), odds["got_bye"])],
    "bye: top 2 now": [brier(odds["now_top2"], odds["got_bye"])],
}, index=["all weeks"])
pd.concat([by_week, overall]).round(3)

In [ ]:
# Uncertainty: resample the 12 teams (each team's 12 predictions stay together), 5,000 times.
rng = np.random.default_rng(2025)
teams = odds["roster_id"].unique()
groups = {r: g for r, g in odds.groupby("roster_id")}
draws = []
for _ in range(5000):
    sample = pd.concat([groups[r] for r in rng.choice(teams, size=len(teams), replace=True)])
    model = brier(sample["p_playoffs"], sample["made_playoffs"])
    draws.append((model, model - brier(sample["now_top6"], sample["made_playoffs"]), model - 0.25))
draws = np.array(draws)
pd.DataFrame(np.percentile(draws, [2.5, 50, 97.5], axis=0).T, index=["model Brier", "model − top 6 now", "model − everyone 50%"],
             columns=["2.5%", "median", "97.5%"]).round(3)

## 4. Reliability table

In [ ]:
bins = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0000001]
labels = ["0–10%", "10–30%", "30–50%", "50–70%", "70–90%", "90–100%"]
odds["bin"] = pd.cut(odds["p_playoffs"], bins=bins, labels=labels, right=False)
reliability = odds.groupby("bin", observed=False).agg(
    predictions=("p_playoffs", "size"), teams=("roster_id", "nunique"),
    mean_predicted=("p_playoffs", "mean"), made_playoffs=("made_playoffs", "mean"))
reliability[["mean_predicted", "made_playoffs"]] = reliability[["mean_predicted", "made_playoffs"]].mul(100).round(0)
reliability  # mean predicted vs actual share, in %; 'teams' shows how few independent outcomes sit behind each row

In [ ]:
import plotly.graph_objects as go

fig = go.Figure([
    go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="#9AA79F", dash="dot"), name="perfectly calibrated"),
    go.Scatter(x=reliability["mean_predicted"], y=reliability["made_playoffs"], mode="markers+text",
               marker=dict(size=np.sqrt(reliability["predictions"].clip(lower=1)) * 4, color="#E8590C"),
               text=reliability["predictions"].astype(str), textposition="top center", name="2025, weeks 3–14"),
])
fig.update_layout(width=520, height=520, xaxis_title="Predicted playoff odds (%)", yaxis_title="Made the playoffs (%)",
                  xaxis_range=[-5, 105], yaxis_range=[-5, 105], template="simple_white",
                  title="Reliability: label = number of predictions")
fig.show()

## 5. Sensitivity to `shrink_weeks` (in-sample)

In [ ]:
rows = []
for k in [2, 4, 6, 9, 12, 20]:
    alt = po.build_playoff_odds(team_weeks, schedule, league, {**PARAMS, "shrink_weeks": k})
    alt = alt.join(final[["made_playoffs", "got_bye"]], on="roster_id")
    rows.append({"shrink_weeks": k, "playoffs Brier": brier(alt["p_playoffs"], alt["made_playoffs"]),
                 "bye Brier": brier(alt["p_bye"], alt["got_bye"]),
                 "weeks 3–7 playoffs Brier": brier(alt.loc[alt["week"] <= 7, "p_playoffs"], alt.loc[alt["week"] <= 7, "made_playoffs"])})
pd.DataFrame(rows).set_index("shrink_weeks").round(4)